# Initialized package

In [ ]:
!pip install -q kaggle cryptography

In [ ]:
from cryptography.fernet import Fernet
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC as Derived
from cryptography.hazmat.backends import default_backend
from getpass import getpass
import zipfile
import base64
import pandas as pd
import os
import json
from pathlib import Path

# The Keys

In [ ]:
def ForKeys(password: str, salt: bytes) -> bytes:
    kdf = Derived(algorithm=hashes.SHA256(),
                length=32,
                salt = salt,
                iterations=100000,
                backend=default_backend(),
                )
    key = base64.urlsafe_b64encode(kdf.derive(password.encode()))
    return key

In [ ]:
def DecoderPass(Enkripsi: str, password: str = None) -> str:
    if password is None:
        password = getpass("Password: ")
    decoded_data = base64.urlsafe_b64decode(Enkripsi.encode())
    salt = decoded_data[:16]
    encrypted_data = decoded_data[16:]
    key = ForKeys(password, salt)
    f = Fernet(key)
    decrypted_data = f.decrypt(encrypted_data)
    Hasil = decrypted_data.decode()
    return Hasil

In [ ]:
username_Encripted = '5uvtqVoPdK6BDPnu5gT_zWdBQUFBQUJwT0F6U1BHMl9OQUYtY2VSOE5KS3lFTFdRWFk3WmFtQk5ORDM2V3FGWUh6TnJNZE1kdnpqNnFzVFMwNDRmbERwSXJSM3RUeFlhakhqZ3JNeHJteXR5SG92TkZRPT0='
APItoken_Encripted = '7rxcrYPUgIOHRBjYj0O3WWdBQUFBQUJwT0F5a3hQSnhabFRJdjlkRktfcVBTVGxwenFQQWVwc1d5dnVuWHlLRUtOblVFMlUyZU1OOVB5UUpPMEhDdFJ4eXdpWUUzbkh0cHpua1UzUEFKWEhLR1ViaFJaZFBWd3BpbngtaUFCb0dIZTJIZnhmSTNUVFNocmZ5SkpHbDJEQ1hRN1Vo'

In [ ]:
username = DecoderPass(username_Encripted)

Password: ··········


In [ ]:
APItoken = DecoderPass(APItoken_Encripted)

Password: ··········


In [ ]:
def KaggleSetup(username, Token):
    Success = False
    try :
        auth = {'username': username, 'key': Token}
        ConfDir = Path.home() / '.config' / 'kaggle'
        ConfDir.mkdir(parents=True, exist_ok=True)
        Dest = ConfDir / 'kaggle.json'
        with open(Dest, 'w') as thefile:
            json.dump(auth, thefile)
        os.chmod(ConfDir, 0o600)
        print(f'Kaggle API Dir: {ConfDir}')
        Success = True
    except Exception as Arr:
        print(f'Failed : {Arr}')
    finally:
        return Success

def KaggleDown(address : str = 'andrexibiza/grocery-sales-dataset'):
    Success = False
    try :
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        print(f'Downloading {address} ...', end = '\r')
        api.dataset_download_files(address, path='.', unzip=True)
        print('Download Success')
        Success = True
    except Exception as Arr:
        print(f'Failed : {Arr}')
    finally:
        print('Files:', os.listdir('.'))
        return Success

# InstaCart Superstore Dataset

## Initialized

In [ ]:
InstaCartPath = "mohdshahnawazaadil/supermarket-superstore-dataset-bundle"

InstaDir = '/content/working/InstaDir'
KaggleSetup(username, APItoken)
os.makedirs(InstaDir, exist_ok=True)
os.chdir(InstaDir)
Status = KaggleDown(InstaCartPath)
print(Status)

Kaggle API Dir: /root/.config/kaggle
Dataset URL: https://www.kaggle.com/datasets/mohdshahnawazaadil/supermarket-superstore-dataset-bundle
Download Success
Files: ['aisles.csv', 'order_products__train.csv', 'orders.csv', 'products.csv', 'departments.csv', 'sample_submission.csv', 'order_products__prior.csv']
True


## Aisles of Product

In [ ]:
from duckdb import connect as dcon

ic = dcon('InstaCart.db')
ic.execute('CREATE SCHEMA IF NOT EXISTS InstaCart_Sales')

ic.execute("CREATE OR REPLACE TABLE AISLE AS SELECT * FROM 'aisles.csv'")

In [ ]:
AisleData = ic.execute("PRAGMA table_info('AISLE')").fetchdf()
display(AisleData)

,cid,name,type,notnull,dflt_value,pk
0,0,aisle_id,BIGINT,False,None,False
1,1,aisle,VARCHAR,False,None,False


In [ ]:
JustSampling = '''
SELECT *
FROM AISLE
TABLESAMPLE 5 ROWS;
'''

aisles = ic.execute(JustSampling).fetchdf()
display(aisles)

,aisle_id,aisle
0,20,oral hygiene
1,27,beers coolers
2,39,seafood counter
3,43,buns rolls
4,107,chips pretzels


## Order Train Data

In [ ]:
ic.execute("CREATE OR REPLACE TABLE OrderTrain AS SELECT * FROM 'order_products__train.csv'")
OPTData = ic.execute("PRAGMA table_info('OrderTrain')").fetchdf()
display(OPTData)

,cid,name,type,notnull,dflt_value,pk
0,0,order_id,BIGINT,False,None,False
1,1,product_id,BIGINT,False,None,False
2,2,add_to_cart_order,BIGINT,False,None,False
3,3,reordered,BIGINT,False,None,False


In [ ]:
JustSampling = '''
SELECT *
FROM OrderTrain
TABLESAMPLE 5 ROWS;
'''

OPTData = ic.execute(JustSampling).fetchdf()
display(OPTData)

,order_id,product_id,add_to_cart_order,reordered
0,631,7509,16,0
1,18020,29935,2,0
2,303410,13203,28,0
3,313539,31717,11,1
4,328830,5240,12,0


## Order Data

In [ ]:
ic.execute("CREATE OR REPLACE TABLE OrdersDetails AS SELECT * FROM 'orders.csv'")
Order = ic.execute("PRAGMA table_info('OrdersDetails')").fetchdf()
display(Order)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,cid,name,type,notnull,dflt_value,pk
0,0,order_id,BIGINT,False,None,False
1,1,user_id,BIGINT,False,None,False
2,2,eval_set,VARCHAR,False,None,False
3,3,order_number,BIGINT,False,None,False
4,4,order_dow,BIGINT,False,None,False
5,5,order_hour_of_day,VARCHAR,False,None,False
6,6,days_since_prior_order,DOUBLE,False,None,False


In [ ]:
JustSampling = '''
SELECT *
FROM OrdersDetails
TABLESAMPLE 5 ROWS;
'''

Order = ic.execute(JustSampling).fetchdf()
display(Order)

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2839132,155,prior,21,5,02,30.0
1,62412,311,prior,9,1,09,14.0
2,3231904,7678,prior,4,2,15,25.0
3,3420706,7815,prior,3,5,14,26.0
4,3409351,8134,test,6,1,14,30.0


## Product Database

In [ ]:
from duckdb import Error as dErr

def DataLoad(Conns : object,
             fpath : str,
             table_name: str = "Orders"):
    INPO, SAMPLE = None, None
    try:
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"The file at {fpath} was not found.")

        Query01 = f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM '{fpath}'"
        Conns.execute(Query01)

        Query02 = f"PRAGMA table_info('{table_name}')"
        Query03 = f"SELECT * FROM {table_name} TABLESAMPLE 5 ROWS"
        INPO = Conns.execute(Query02).fetchdf()
        SAMPLE = Conns.execute(Query03).fetchdf()

        print(f"\n--- {table_name} Schema ---")
        display(INPO)
        print(f"\n--- {table_name} 5-Row Sample ---")
        display(SAMPLE)
    except FileNotFoundError as e:
        print(f"File System Error: {e}")
    except dErr as e:
        print(f"DuckDB Error: {e}")
    except Exception as e:
        print(f"Unexpected Error: {e}")
    finally:
        return INPO, SAMPLE

In [ ]:
_, ProductsData = DataLoad(ic, 'products.csv', 'ProductsData')


--- ProductsData Schema ---


,cid,name,type,notnull,dflt_value,pk
0,0,product_id,BIGINT,False,None,False
1,1,product_name,VARCHAR,False,None,False
2,2,aisle_id,BIGINT,False,None,False
3,3,department_id,BIGINT,False,None,False



--- ProductsData 5-Row Sample ---


,product_id,product_name,aisle_id,department_id
0,4129,Mighty Veggie Carrot Pear Pomegranate & Oats V...,92,18
1,13673,Organic Cheddar Snack Mix Bunnies,78,19
2,16146,Dark Chocolate Covered Salted Almonds,45,19
3,17540,Burnt Sugar Vanilla Ice Cream,37,1
4,19934,Fat Free Smooth & Creamy Plain Organic Yogurt,120,16


## Department Data

In [ ]:
_, DepartmentData = DataLoad(ic, 'departments.csv', 'DepartmentData')


--- DepartmentData Schema ---


,cid,name,type,notnull,dflt_value,pk
0,0,department_id,BIGINT,False,None,False
1,1,department,VARCHAR,False,None,False



--- DepartmentData 5-Row Sample ---


,department_id,department
0,2,other
1,13,pantry
2,16,dairy eggs
3,18,babies
4,21,missing


## Order Test Data

In [ ]:
_, OrderTest = DataLoad(ic, 'order_products__prior.csv', 'OrderTest')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- OrderTest Schema ---


,cid,name,type,notnull,dflt_value,pk
0,0,order_id,BIGINT,False,None,False
1,1,product_id,BIGINT,False,None,False
2,2,add_to_cart_order,BIGINT,False,None,False
3,3,reordered,BIGINT,False,None,False



--- OrderTest 5-Row Sample ---


,order_id,product_id,add_to_cart_order,reordered
0,875,2825,4,1
1,1105,24964,11,0
2,1319,46107,14,1
3,1776,30391,6,1
4,1960,6740,12,0


## List of Table in Database

In [ ]:
tables = ic.execute("PRAGMA show_tables;").fetchdf()
display(tables)

,name
0,AISLE
1,DepartmentData
2,OrderTest
3,OrderTrain
4,OrdersDetails
5,ProductsData


## Training Data Unification

### Checking Number of Data

In [ ]:
Query01 = 'Select count(*) from OrderTrain'
Count01 = ic.execute(Query01).fetchone()[0]
print(f'Number of Rows from OrderTrain is {Count01:,} rows.')

Number of Rows from OrderTrain is 1,384,617 rows.


In [ ]:
Query02 = 'Select count(*) from OrdersDetails'
Count02 = ic.execute(Query02).fetchone()[0]
print(f'Number of Rows from OrdersDetails is {Count02:,} rows.')

Number of Rows from OrdersDetails is 3,421,083 rows.


In [ ]:
Query03 = 'Select count(*) from ProductsData;'
Count03 = ic.execute(Query03).fetchone()[0]
print(f'Number of Rows from ProductsData is {Count03:,} rows.')

Number of Rows from ProductsData is 49,688 rows.


In [ ]:
Query04 = '''
WITH X AS (
SELECT DISTINCT
    A1.order_id,
    A1.user_id,
    A1.eval_set,
    A1.order_number,
    A1.order_dow,
    A1.order_hour_of_day,
    A1.days_since_prior_order,
    A2.product_id,
    A2.add_to_cart_order,
    A2.reordered,
FROM
    OrdersDetails AS A1
LEFT JOIN
    OrderTrain AS A2
    ON A1.order_id = A2.order_id
ORDER BY
    A1.order_id, A2.add_to_cart_order),

X1 AS (
SELECT
    A1.order_id,
    A1.user_id,
    A1.eval_set,
    A1.order_number,
    A1.order_dow,
    A1.order_hour_of_day,
    A1.days_since_prior_order,
    A2.product_id,
    A2.add_to_cart_order,
    A2.reordered,
FROM
    OrdersDetails AS A1
INNER JOIN
    OrderTrain AS A2
    ON A1.order_id = A2.order_id
ORDER BY
    A1.order_id, A2.add_to_cart_order),

Y AS (
SELECT
    A.*,
    A3.aisle_id,
    A3.department_id
FROM
    X AS A
LEFT JOIN
    ProductsData AS A3
    ON A.product_id = A3.product_id
),

Y01 AS (
SELECT *
FROM
    OrderTrain AS A2
LEFT JOIN
    ProductsData AS A3
    ON A2.product_id = A3.product_id
),

Z1 AS (
SELECT
    'Order Details + Order Training (Left Join)' AS CATEGORY,
    COUNT(*) AS TOTAL
FROM
    X
),

Z2 AS (
SELECT
    'Train + ProductDetail' AS CATEGORY,
    COUNT(*) AS TOTAL
FROM
    Y01
),

Z3 AS (
SELECT
    'All Joinning (Left Join)' AS CATEGORY,
    COUNT(*) AS TOTAL
FROM
    Y
),

Z4 AS (
SELECT
    'Order Details + Order Training (Inner Join)' AS CATEGORY,
    COUNT(*) AS TOTAL
FROM
    X1
)

SELECT * FROM Z1
UNION ALL
SELECT * FROM Z4
UNION ALL
SELECT * FROM Z2
UNION ALL
SELECT * FROM Z3;
'''

In [ ]:
Count04 = ic.execute(Query04).fetchdf()
display(Count04)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,CATEGORY,TOTAL
0,Order Details + Order Training (Left Join),4674491
1,Order Details + Order Training (Inner Join),1384617
2,Train + ProductDetail,1384617
3,All Joinning,4674491


In [ ]:
Query05 = '''
WITH OrderAggregates AS (
    SELECT
        A2.order_id,
        -- Aggregate product information into arrays/strings
        ARRAY_AGG(A2.product_id) AS product_ids,
        ARRAY_AGG(A2.add_to_cart_order) AS add_to_cart_orders,
        ARRAY_AGG(A2.reordered) AS reordered_flags,
        ARRAY_AGG(A3.aisle_id) AS aisle_ids,
        ARRAY_AGG(A3.department_id) AS department_ids,
        COUNT(*) AS total_products_in_order
    FROM OrderTrain AS A2
    LEFT JOIN ProductsData AS A3
        ON A2.product_id = A3.product_id
    GROUP BY order_id
),

FinalResult AS (
    SELECT
        A1.order_id,
        A1.user_id,
        A1.eval_set,
        A1.order_number,
        A1.order_dow,
        A1.order_hour_of_day,
        A1.days_since_prior_order,
        -- Product aggregates (will be NULL for orders not in OrderTrain)
        OA.product_ids,
        OA.add_to_cart_orders,
        OA.reordered_flags,
        OA.aisle_ids,
        OA.department_ids,
        OA.total_products_in_order
    FROM OrdersDetails AS A1
    LEFT JOIN OrderAggregates AS OA
        ON A1.order_id = OA.order_id
)

SELECT
    'FINAL_RESULT' AS CATEGORY,
    COUNT(*) AS TOTAL
FROM FinalResult;
'''

Count05 = ic.execute(Query05).fetchdf()
display(Count05)

,CATEGORY,TOTAL
0,FINAL_RESULT,3421083


### Finalized the Training Query

In [ ]:
Query06 = '''
WITH AGGREGATION AS (
    SELECT
        A2.order_id,
        ARRAY_AGG(A2.product_id) AS product_ids,
        ARRAY_AGG(A2.add_to_cart_order) AS add_to_cart_orders,
        ARRAY_AGG(A2.reordered) AS reordered_flags,
        ARRAY_AGG(A3.aisle_id) AS aisle_ids,
        ARRAY_AGG(A3.department_id) AS department_ids,
        COUNT(*) AS total_products_in_order
    FROM
        OrderTrain AS A2
    LEFT JOIN
        ProductsData AS A3
    ON
        A2.product_id = A3.product_id
    GROUP BY
        order_id
),

DETAILS AS (
    SELECT
        A1.order_id,
        A1.user_id,
        A1.eval_set,
        A1.order_number,
        A1.order_dow,
        A1.order_hour_of_day,
        A1.days_since_prior_order,
        OA.product_ids,
        OA.add_to_cart_orders,
        OA.reordered_flags,
        OA.aisle_ids,
        OA.department_ids,
        OA.total_products_in_order
    FROM
        OrdersDetails AS A1
    LEFT JOIN
        AGGREGATION AS OA
    ON
        A1.order_id = OA.order_id
    ORDER BY
        A1.order_id ASC
)

SELECT
    *
FROM
    DETAILS;
'''

Data06 = ic.execute(Query06).fetchdf()
display(Data06.sample(8))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_ids,add_to_cart_orders,reordered_flags,aisle_ids,department_ids,total_products_in_order
844675,844676,180332,prior,1,5,13,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3024852,3024853,9370,prior,3,3,10,30.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
647694,647695,416,prior,2,1,05,30.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2475050,2475051,198240,train,65,0,11,5.0,"[13176, 21292, 21137, 30777, 43352, 8571]","[1, 2, 3, 4, 5, 6]","[1, 1, 1, 1, 1, 1]","[24, 32, 24, 67, 32, 32]","[4, 4, 4, 20, 4, 4]",6
2533516,2533517,133332,prior,4,4,20,25.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2089367,2089368,116388,test,34,1,23,8.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
180019,180020,137026,prior,17,0,07,30.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
206434,206435,163671,prior,11,6,11,28.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [ ]:
Query07 = '''
CREATE OR REPLACE
    TABLE FullTrainData AS
WITH
AGGREGATION AS (
SELECT DISTINCT
    A1.order_id,
    A1.user_id,
    A1.eval_set,
    A1.order_number,
    A1.order_dow,
    A1.order_hour_of_day,
    A1.days_since_prior_order,
    A2.product_id,
    A2.add_to_cart_order,
    A2.reordered,
    A3.aisle_id,
    A3.department_id
FROM
    OrdersDetails AS A1
INNER JOIN
    OrderTrain AS A2
    ON A1.order_id = A2.order_id
LEFT JOIN
    ProductsData AS A3
    ON A2.product_id = A3.product_id
ORDER BY
    A1.order_id, A2.add_to_cart_order),

FILTER AS (
    SELECT
        *,
        CASE
            WHEN eval_set = 'train' AND product_id IS NOT NULL
            THEN 1
            ELSE 0
        END AS IsTrain
    FROM
        AGGREGATION
    WHERE
        IsTrain = 1)

SELECT
    order_id,
    user_id,
    product_id,
    aisle_id,
    department_id,
    order_number,
    order_dow,
    order_hour_of_day,
    days_since_prior_order,
    add_to_cart_order,
    reordered
FROM
    FILTER
ORDER BY
    order_id ASC, Product_id ASC;
'''

ic.execute(Query07)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
Check02 = ic.execute("SELECT * FROM FullTrainData TABLESAMPLE 7 ROWS;").fetchdf()
display(Check02)

,order_id,user_id,product_id,aisle_id,department_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,add_to_cart_order,reordered
0,13248,62255,44267,94,7,12,6,11,14.0,19,1
1,17912,38255,27344,96,20,4,6,19,6.0,8,0
2,41669,93211,18840,45,19,35,1,07,6.0,3,1
3,53119,65906,47000,54,17,23,5,15,5.0,13,1
4,54804,2499,43968,88,13,4,1,16,30.0,30,0
5,63889,185292,35923,5,13,7,0,13,4.0,7,0
6,66358,74973,6878,28,5,4,1,17,28.0,5,0


In [ ]:
header = Check02.columns.tolist()
print(header)

['order_id', 'user_id', 'product_id', 'aisle_id', 'department_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'add_to_cart_order', 'reordered']


In [ ]:
tables = ic.execute("PRAGMA show_tables;").fetchdf()
display(tables)

,name
0,AISLE
1,DepartmentData
2,FullTrainData
3,OrderTest
4,OrderTrain
5,OrdersDetails
6,ProductsData


### Save the Database

In [ ]:
os.getcwd()

'/content/working/InstaDir'

In [ ]:
ic.close()
print('DuckDB connection closed and database saved.')

DuckDB connection closed and database saved.


In [ ]:
from google.colab import drive
from shutil import copy2

drive.mount('/content/drive')
source_path = os.path.join(os.getcwd(), './InstaCart.db')
Dest = '/content/drive/My Drive/Colab Notebooks'

os.makedirs(Dest, exist_ok=True)
copy2(source_path, Dest)

print(f"'{source_path}' successfully exported to '{Dest}'")

Mounted at /content/drive
'/content/working/InstaDir/./InstaCart.db' successfully exported to '/content/drive/My Drive/Colab Notebooks'
